# 代名詞が何を指すかを常識で判断させる (Winograd Schema Challenge)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/19_semantics/winograd_ja.ipynb)

NLP クックブック 19-05 ／ 意味解析

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

「狙撃兵はテロリストを撃った。彼は悪漢だったからだ。」の「彼」が誰を指すかのように、文法だけでは決まらず、常識が必要な照応の問題をモデルに解かせたい。

In [1]:
import re, urllib.request
import numpy as np
import pandas as pd
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

url = "https://raw.githubusercontent.com/ku-nlp/Winograd-Schema-Challenge-Ja/master/test.txt"
blocks = urllib.request.urlopen(url).read().decode("utf-8").strip().split("\n\t\t\n")
problems = []
for block in blocks:
    lines = [line.split("\t") for line in block.split("\n")]
    if len(lines) >= 4:
        candidates = [c.strip() for c in re.split(r"[、,，]", lines[2][1]) if c.strip()]
        problems.append({"text": lines[0][1], "pronoun": lines[1][1], "candidates": candidates, "answer": lines[3][1].strip()})
problems = [p for p in problems if len(p["candidates"]) == 2 and p["answer"] in p["candidates"]]
print(len(problems), "問"); print(problems[0]); print(problems[1])

563 問
{'text': '狙撃兵はテロリストを撃った。彼は悪漢だったからだ。', 'pronoun': '彼', 'candidates': ['狙撃兵', 'テロリスト'], 'answer': 'テロリスト'}
{'text': '狙撃兵はテロリストを撃った。彼は命令されたからだ。', 'pronoun': '彼', 'candidates': ['狙撃兵', 'テロリスト'], 'answer': '狙撃兵'}


In [2]:
name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す（特殊トークンやチャットテンプレートはそのまま）
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto").eval()
ONE_TWO = [tokenizer(c, add_special_tokens=False)["input_ids"][-1] for c in ["1", "2"]]

@torch.no_grad()
def p_first(text, pronoun, a, b):
    prompt = (f"次の文の「{pronoun}」が指しているのはどちらですか。番号だけで答えてください。\n\n文: {text}\n1. {a}\n2. {b}")
    chat = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, tokenize=False)
    ids = tokenizer(chat, add_special_tokens=False, return_tensors="pt")["input_ids"].to(llm.device)
    return float(llm(ids).logits[0, -1, ONE_TWO].float().softmax(-1)[0])

def solve(p):
    """選択肢の順序を入れ替えて 2 回聞き、平均する（1 番目を選びやすい、といった偏りを打ち消す）"""
    a, b = p["candidates"]
    score = (p_first(p["text"], p["pronoun"], a, b) + 1 - p_first(p["text"], p["pronoun"], b, a)) / 2
    return a if score > 0.5 else b

sample = problems[:200]
df = pd.DataFrame(sample)
df["prediction"] = [solve(p) for p in sample]   # 1 問に 2 回ずつ計算する。T4 なら 200 問で 1〜2 分
df["correct"] = df["prediction"] == df["answer"]
print("正解率:", round(df["correct"].mean(), 3), "(でたらめなら 0.5)")

正解率: 0.64 (でたらめなら 0.5)


In [3]:
# このデータは 2 文で 1 組（同じ登場人物で、1 語か 2 語だけ違い、答えが入れ替わる）。組の両方に正解できた割合を見る
pair_ok = [df["correct"][i] and df["correct"][i + 1] for i in range(0, len(df) - 1, 2)]
print("組の両方に正解:", round(np.mean(pair_ok), 3), "/ 常に同じ側を答えているだけの組:",
      round(np.mean([df["prediction"][i] == df["prediction"][i + 1] for i in range(0, len(df) - 1, 2)]), 3))
print("先に出てくる候補を選んだ割合:", round(np.mean([p == c[0] for p, c in zip(df["prediction"], df["candidates"])]), 3))
for r in df[~df["correct"]].head(5).itertuples():
    print(f"× {r.text}\n     「{r.pronoun}」→ 予測: {r.prediction} / 正解: {r.answer}")

組の両方に正解: 0.39 / 常に同じ側を答えているだけの組: 0.16
先に出てくる候補を選んだ割合: 0.545
× ビルはボブの顔を殴った。彼はメアリに失礼な態度をとったからだ。
     「彼」→ 予測: ビル / 正解: ボブ
× ビルはボブと遊ぶのが好きだ。しかし彼はそうしたくなかった。
     「彼」→ 予測: ビル / 正解: ボブ
× 販売員はデイブに言った。この車は彼がこれまで販売したたうちで一番良い車となるでしょう。
     「彼」→ 予測: デイブ / 正解: 販売員
× その学生は教授に助けてもらいたかったがやめた。彼は非常に威圧だったからだ。
     「彼」→ 予測: 学生 / 正解: 教授
× フランシスは立ち止まってペンギンを見た。彼が可愛らしかったからだ。
     「彼」→ 予測: フランシス / 正解: ペンギン


- Winograd Schema Challenge は、文法的にはどちらの候補も先行詞になれるが、常識を使えば一方に決まる問題を集めたもの。1 語を変えると答えが入れ替わる 2 文が組になっている。
- 組になっていることが重要。片方だけ正解しても、「主語を選びやすい」「先に出たほうを選びやすい」といった癖で当たっただけかもしれない。組の両方に正解した割合のほうが、実力をよく表す。
- 日本語版 (JWSC) は英語からの翻訳で作られている。日本語ではゼロ代名詞が自然な場面でも「彼」「それ」が使われるので、やや不自然な文が含まれる。
- 事前学習済みモデルの登場で、この種の問題の正解率は大きく上がった。ただし、Web 上の類似文を覚えているだけの可能性や、データの偏りを使っている可能性が指摘されており、改良版（WinoGrande など）が作られている。
- 照応解析の一般的な方法は、この章の「日本語の共参照解析をする (KWJA)」を参照。